# Experiment 1 — NCF Hyperparameter Tuning (MovieLens 1M)

**STAT 238 · Final Project · UC Berkeley Spring 2026**

---

## Setup

A concurrent CS 289A project compares matrix factorisation, NCF, and a two-stage ranker
on MovieLens 1M to study how recommendation quality degrades as training data sparsity
increases. NCF has five hyperparameters that must be chosen before the sparsity sweep
can be run fairly:

| Parameter | Symbol | Range | Encoding for GP |
|---|---|---|---|
| Embedding dim | $d$ | $\{32,64,128,256\}$ | ordinal $\in[0,3]$ |
| MLP config | — | 3 options | ordinal $\in[0,2]$ |
| Learning rate | $\eta$ | $[10^{-4}, 10^{-2}]$ | $\log\eta \in[-9.21,-4.61]$ |
| L2 weight decay | $\lambda$ | $[10^{-6}, 10^{-3}]$ | $\log\lambda \in[-13.82,-6.91]$ |
| WMF scale | $\alpha$ | $[0.5, 5.0]$ | as-is |

**Objective**: validation NDCG@10 — higher is better.  
**Budget**: 28 trials × ~4 min/trial on SCF GPU ≈ 2 hours.  
**Baselines**: random search and grid search at matched budgets.

## Why log-transform $\eta$ and $\lambda$?

The SE kernel assigns distances as $\|x - x'\|^2$.  Without log-transforming,
the gap between $\eta = 10^{-4}$ and $\eta = 10^{-3}$ is $9\times10^{-4}$,
while the gap between $\eta = 10^{-3}$ and $\eta = 10^{-2}$ is $9\times10^{-3}$ —
a 10× disparity in Euclidean distance for the same multiplicative step.
In log-space both gaps equal $\log 10 \approx 2.30$, so the GP kernel treats
them symmetrically, as it should.

---

## How the compute was run

The expensive part (calling `train.py` 28× on an SCF GPU node) was run via:

```bash
# on SCF login node
srun --partition=jsteinhardt --gres=gpu:1 --mem=32G --time=06:00:00 --pty bash

# once on the GPU node
conda activate cs289-ranking-gpu
cd ~/Bayesian-Optimization-for-ML
python jobs/run_bo_ncf.py
```

Results were saved to `results/ncf/trials.csv` after every trial.  
**This notebook loads that CSV — all cells run without a GPU.**

In [ ]:
import sys, math, os
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.gp import GaussianProcess
from src.acquisition import expected_improvement
from src.black_box_ncf import NCF_PARAM_SPACE, decode_params

plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "figure.dpi": 120})

N_INIT = 5   # random initialisations — used for phase labelling in plots

# Param space bounds (for GP rebuild)
LO = np.array([p["bounds"][0] for p in NCF_PARAM_SPACE])
HI = np.array([p["bounds"][1] for p in NCF_PARAM_SPACE])
PARAM_NAMES = [p["name"] for p in NCF_PARAM_SPACE]   # encoded names saved in CSV
print("Imports OK.")

## 1  Load Results

In [ ]:
# ---- BO trials ----
bo_df = pd.read_csv("../results/ncf/trials.csv")
bo_df["best_so_far"] = bo_df["val_ndcg"].cummax()

print(f"BO trials loaded   : {len(bo_df)}")
print(f"Best NDCG@10       : {bo_df['val_ndcg'].max():.5f}")
print(f"Mean NDCG@10       : {bo_df['val_ndcg'].mean():.5f}")
print(f"Mean runtime/trial : {bo_df['runtime_s'].mean():.0f}s  "
      f"({bo_df['runtime_s'].mean()/60:.1f} min)")
print()
bo_df.head()

In [ ]:
# ---- Random search trials ----
rs_df = pd.read_csv("../results/ncf_random/trials.csv")
rs_df["best_so_far"] = rs_df["val_ndcg"].cummax()
print(f"Random search trials loaded: {len(rs_df)}")

# ---- Grid search trials ----
grid_df = pd.read_csv("../results/ncf_grid/trials.csv")
grid_df["best_so_far"] = grid_df["val_ndcg"].cummax()
print(f"Grid search trials loaded  : {len(grid_df)}")

## 2  Convergence Curves

Best NDCG@10 observed so far vs trial number — the main comparison figure.

In [ ]:
n = min(len(bo_df), len(rs_df), len(grid_df))
trials = np.arange(1, n + 1)

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(trials, bo_df["best_so_far"].values[:n],
        color="tab:blue",  lw=2.5, label="Bayesian Optimisation (GP + EI)")
ax.plot(trials, rs_df["best_so_far"].values[:n],
        color="tab:red",   lw=2.5, ls="--", label="Random Search")
ax.plot(trials, grid_df["best_so_far"].values[:n],
        color="tab:green", lw=2.5, ls=":",  label="Grid Search ($2^5$)")

ax.axvline(N_INIT + 0.5, color="gray", lw=1, ls=":", alpha=0.7)
ax.text(N_INIT + 0.7, ax.get_ylim()[0] + 0.002, "BO starts", fontsize=9, color="gray")

ax.set_xlabel("Trial number")
ax.set_ylabel("Best val NDCG@10 observed so far")
ax.set_title(
    "NCF hyperparameter tuning — MovieLens 1M\n"
    "Convergence curves: BO vs Random vs Grid Search"
)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.4f"))
plt.tight_layout()
plt.savefig("../figures/ncf_convergence.png", bbox_inches="tight")
plt.show()

print(f"\nFinal best — BO: {bo_df['val_ndcg'].max():.5f}  "
      f"RS: {rs_df['val_ndcg'].max():.5f}  "
      f"Grid: {grid_df['val_ndcg'].max():.5f}")

## 3  Trial-by-Trial View

In [ ]:
bo_phase  = bo_df[bo_df["trial"] > N_INIT]
rnd_phase = bo_df[bo_df["trial"] <= N_INIT]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: NDCG@10 per trial
ax = axes[0]
ax.scatter(rnd_phase["trial"], rnd_phase["val_ndcg"],
           c="tab:orange", s=70, zorder=4, label="Random init")
ax.scatter(bo_phase["trial"], bo_phase["val_ndcg"],
           c="tab:blue",   s=70, zorder=4, label="BO trials")
ax.plot(bo_df["trial"], bo_df["best_so_far"],
        color="black", lw=1.5, ls="--", label="Best so far")
ax.axvline(N_INIT + 0.5, color="gray", lw=1, ls=":")
ax.set_xlabel("Trial"); ax.set_ylabel("val NDCG@10")
ax.set_title("NDCG@10 per trial")
ax.legend(fontsize=9)

# Histogram
ax2 = axes[1]
ax2.hist(rnd_phase["val_ndcg"], bins=6, alpha=0.65,
         color="tab:orange", label=f"Random init (n={len(rnd_phase)})")
ax2.hist(bo_phase["val_ndcg"],  bins=6, alpha=0.65,
         color="tab:blue",   label=f"BO trials (n={len(bo_phase)})")
ax2.set_xlabel("val NDCG@10"); ax2.set_ylabel("Count")
ax2.set_title("Distribution by phase")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig("../figures/ncf_trial_scatter.png", bbox_inches="tight")
plt.show()

## 4  Trial History — Top 10

In [ ]:
top10 = (bo_df
         .sort_values("val_ndcg", ascending=False)
         .reset_index(drop=True))
top10.index += 1

print("Top 10 BO trials by NDCG@10:")
print(top10[["trial","emb_dim","mlp_layers","lr","l2","alpha",
             "val_ndcg","hr","runtime_s"]].head(10).to_string())

## 5  GP Surrogate — Final State

Rebuild the GP on all 28 observations and visualise the posterior mean and EI
as a 2D slice through the $\eta$–$\lambda$ plane (holding `emb_dim`, `mlp_layers`,
and $\alpha$ fixed at the best found values).

Posterior formulas from Lecture 23:

$$\mu(x) = \mathbf{k}(x)^\top (K + \sigma^2 I_n)^{-1} y \qquad [\text{Lec 23, Eq. 2}]$$

$$v(x) = K(x,x) - \mathbf{k}(x)^\top (K + \sigma^2 I_n)^{-1} \mathbf{k}(x) \qquad [\text{Lec 23, Sec 2}]$$

In [ ]:
# Reconstruct encoded X matrix from saved CSV columns
# CSV stores: emb_dim_x, mlp_x, log_lr, log_l2  (encoded) + alpha (= actual, same scale)
X_all_raw = bo_df[["emb_dim_x", "mlp_x", "log_lr", "log_l2", "alpha"]].values  # (n, 5)
y_all     = bo_df["val_ndcg"].values

# Normalise to [0,1]^5  (GP always sees unit-cube inputs)
X_unit = (X_all_raw - LO) / (HI - LO)
y_std  = (y_all - y_all.mean()) / y_all.std()

gp_final = GaussianProcess(n_restarts=5)
gp_final.condition(X_unit, y_std)
print(f"Final GP: {gp_final}")

# Best encoded config from the CSV
best_row = bo_df.loc[bo_df["val_ndcg"].idxmax()]
best_enc_arr = np.array([
    best_row["emb_dim_x"], best_row["mlp_x"],
    best_row["log_lr"],    best_row["log_l2"],
    best_row["alpha"]
])
best_unit = (best_enc_arr - LO) / (HI - LO)

# Build 60×60 grid over log_lr (dim 2) and log_l2 (dim 3)
n_grid = 60
lr_range = np.linspace(0, 1, n_grid)
l2_range = np.linspace(0, 1, n_grid)
LR_g, L2_g = np.meshgrid(lr_range, l2_range)

base = np.tile(best_unit, (n_grid * n_grid, 1))
base[:, 2] = LR_g.ravel()
base[:, 3] = L2_g.ravel()

mu_grid, _  = gp_final.predict(base)
ei_grid     = expected_improvement(base, gp_final, y_best=y_std.max())

# Back to original scale for axis labels
lr_orig = np.exp(LO[2] + lr_range * (HI[2] - LO[2]))
l2_orig = np.exp(LO[3] + l2_range * (HI[3] - LO[3]))
LR_orig, L2_orig = np.meshgrid(lr_orig, l2_orig)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
best_lr_val = np.exp(best_row["log_lr"])
best_l2_val = np.exp(best_row["log_l2"])

for ax, Z, label in [
    (axes[0], mu_grid.reshape(n_grid, n_grid),
     r"Posterior mean $\mu(x)$ (standardised)"),
    (axes[1], ei_grid.reshape(n_grid, n_grid),
     r"EI$(x)$"),
]:
    ct = ax.contourf(LR_orig, L2_orig, Z, levels=30, cmap="viridis")
    fig.colorbar(ct, ax=ax)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"Learning rate $\eta$")
    ax.set_ylabel(r"L2 weight decay $\lambda$")
    ax.set_title(label)
    ax.plot(best_lr_val, best_l2_val, "r*", markersize=14, label="Best config")
    ax.legend(fontsize=9)

plt.suptitle(
    "GP surrogate — final state (all 28 BO trials)\n"
    r"Slice: $\eta$ vs $\lambda$, other dims at best config",
    y=1.02
)
plt.tight_layout()
plt.savefig("../figures/ncf_gp_final.png", bbox_inches="tight")
plt.show()

## 6  Best Configuration — CS 289A Handoff

The best config found by BO feeds directly into the CS 289A sparsity sweep.
Run these commands on the SCF GPU node at each density level.

In [ ]:
best_enc = {
    "emb_dim_x": best_row["emb_dim_x"],
    "mlp_x":     best_row["mlp_x"],
    "log_lr":    best_row["log_lr"],
    "log_l2":    best_row["log_l2"],
    "alpha":     best_row["alpha"],
}
best = decode_params(best_enc)

print("=" * 60)
print("Best NCF configuration found by BO")
print("=" * 60)
print(f"  emb_dim    : {best['emb_dim']}")
print(f"  mlp_layers : {best['mlp_layers']}")
print(f"  lr         : {best['lr']:.2e}")
print(f"  l2         : {best['l2']:.2e}")
print(f"  alpha      : {best['alpha']:.4f}")
print(f"  val NDCG@10: {best_row['val_ndcg']:.5f}")
print()

mlp_str = " ".join(str(h) for h in best["mlp_layers"])
print("CS 289A sparsity sweep commands:")
print()
for density in [1.0, 0.8, 0.6, 0.4, 0.2]:
    print(f"# density = {density}")
    print(
        f"python src/train.py --model ncf --density {density} "
        f"--emb-dim {best['emb_dim']} --mlp-layers {mlp_str} "
        f"--lr {best['lr']:.2e} --l2 {best['l2']:.2e} "
        f"--alpha {best['alpha']:.4f} --epochs 20 --device cuda"
    )
    print()

## 7  Summary

*(filled in after run)*

| Metric | BO | Random Search | Grid Search |
|---|---|---|---|
| Best val NDCG@10 | — | — | — |
| Trials to reach best RS | — | — | — |
| Total evaluations | 28 | 28 | 28 |

---

**References**  
- Guntuboyina (2026). STAT 238 Lecture 22–23. UC Berkeley.  
- Frazier (2018). A Tutorial on Bayesian Optimization. *arXiv:1807.02811*.  
- Snoek, Larochelle, Adams (2012). Practical BO of ML Algorithms. *NeurIPS 2012*.  
- Harper & Konstan (2015). The MovieLens Datasets. *ACM TIIS*.